# training.ipynb — Master pipeline orchestrator

**Notebook này gọi tuần tự 5 stages để TRAIN toàn bộ pipeline clustering.**

Mỗi cell bash chạy một stage độc lập. Có thể chạy "Run All" để reproduce toàn bộ kết quả từ raw data đến model cluster.

| Stage | Notebook | Mục đích | Time |
|---|---|---|---|
| 0 | `_download_and_split.py` | Tải parquet từ HuggingFace + split 90/10 chunked | ~2 phút |
| 1 | `01_EDA.ipynb` | EDA, sinh 14 biểu đồ | ~3 phút |
| 2 | `02_Cleaning.ipynb` | Clean + chunked apply → `clean_data/*` (17 cột) | ~5 phút |
| 3 | `03_Feature_Engineering.ipynb` | TF-IDF + SVD 150D → `features/X_*.npz` + `X_*_svd.npy` | ~10 phút |
| 3.5 | `_embed_jds.py` | SBERT mpnet 768D embedding (GPU) → `features/X_*_embed.npy` | ~50 phút (GPU), 3-5h (CPU) |
| 4 | `04_Modeling.ipynb` | MiniBatchKMeans + GMM + Optuna + cluster profile | ~15 phút |
| 5 | `05_Embedding_Clustering.ipynb` | SBERT + KMeans + so sánh với stage 4 | ~10 phút |

**Output cuối cùng** trong `models/`:
- `kmeans_best.joblib`, `kmeans_embed_best.joblib` — 2 model KMeans (TF-IDF + SBERT)
- `gmm_best.joblib`, `optuna_study.joblib` — model phụ
- `labels_train.npy`, `labels_test.npy`, `labels_train_embed.npy`, `labels_test_embed.npy`
- `cluster_profiles.csv`, `cluster_names.json` (và `_embed` variants)
- `demo_10_test_samples.csv` (và `_embed` variant)
- `clustering_summary.json`, `clustering_embed_summary.json`

**Hardware requirements:**
- CPU: ≥ 8 cores recommended
- RAM: ≥ 16 GB
- GPU: optional (NVIDIA RTX với CUDA 12.x) — nếu không có, stage 3.5 SBERT mất 3-5h trên CPU
- Disk: ≥ 5 GB free

> **Lưu ý**: notebook này CHỈ để tham khảo / reproduce. KHÔNG cần chạy lúc bảo vệ.
> Lúc bảo vệ, GV chạy `testing.ipynb` (load model + demo 10 mẫu test, < 1 phút).

## Stage 0 — Download data + split 90/10

Tải `data.parquet` (293 MB) từ HuggingFace, chia 90/10 theo `random_state=42`, ghi 2 file CSV (chunked-read tránh OOM).

In [ ]:
import subprocess, sys
result = subprocess.run(['python', 'notebooks/_download_and_split.py'], capture_output=True, text=True, encoding='utf-8')
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Stage 1 — EDA

Khám phá dữ liệu, sinh 14 biểu đồ vào `figures/eda_*.png`. EDA dùng sample 150k vì full 1.1 GB CSV gây OOM khi load pandas.

In [ ]:
import subprocess
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'notebook', '--execute',
     'notebooks/01_EDA.ipynb', '--output', '01_EDA.ipynb'],
    capture_output=True, text=True, encoding='utf-8')
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## Stage 2 — Cleaning

Áp các quyết định cleaning (parse salary thành 3 feature numeric + flag missing, drop sentinel/outlier, normalize categorical, multi-label industries, ...). Chunked-read + write incremental cho full train 545k.

In [ ]:
import subprocess
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'notebook', '--execute',
     'notebooks/02_Cleaning.ipynb', '--output', '02_Cleaning.ipynb'],
    capture_output=True, text=True, encoding='utf-8')
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## Stage 3 — Feature Engineering

- Numeric (6 cols): impute median + StandardScaler.
- OHE categorical (95 cols).
- Multi-label industries (51 cols).
- TF-IDF text (4 cols, ~25k features) — Vietnamese stopwords + max_df=0.85.
- **TruncatedSVD 150D** trên sparse hstack → dense matrix giữ 77.7% variance.
- Output: `X_*.npz` (sparse) + `X_*_svd.npy` (dense).

In [ ]:
import subprocess
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'notebook', '--execute',
     'notebooks/03_Feature_Engineering.ipynb', '--output', '03_Feature_Engineering.ipynb'],
    capture_output=True, text=True, encoding='utf-8')
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## Stage 3.5 — SBERT embedding (GPU)

Encode 605k JD bằng `paraphrase-multilingual-mpnet-base-v2` (768D, multi-lang). Trên GPU RTX 4060 ~50 phút. Output: `features/X_*_embed.npy`.

> **Lưu ý**: stage này CHỈ chạy nếu có GPU NVIDIA. Trên CPU sẽ mất 3-5 giờ.

In [ ]:
import subprocess
result = subprocess.run(['python', 'notebooks/_embed_jds.py'], capture_output=True, text=True, encoding='utf-8')
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## Stage 4 — Clustering trên TF-IDF + SVD (main pipeline)

MiniBatchKMeans k ∈ [3, 25] với 4 metric (inertia/silhouette/DB/CH). Optuna 30 trials tune (k, init, whitening). GMM compare BIC/AIC. Cluster profiling chunked. t-SNE viz. Demo 10 mẫu test.

**Kết quả**: k=5, silhouette = 0.128, Davies-Bouldin = 2.21.

In [ ]:
import subprocess
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'notebook', '--execute',
     'notebooks/04_Modeling.ipynb', '--output', '04_Modeling.ipynb'],
    capture_output=True, text=True, encoding='utf-8')
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## Stage 5 — Clustering trên SBERT embeddings (comparison)

MiniBatchKMeans trên SBERT 768D L2-normalized. So sánh metric với stage 4. Cluster profiling + t-SNE + demo + bảng so sánh.

**Kết quả**: SBERT silhouette = 0.068 (thấp hơn stage 4 = 0.128). Finding: trên dataset JD VN không fine-tune, TF-IDF lexical features cho clustering tốt hơn SBERT semantic. SBERT phù hợp similarity search/classification hơn raw clustering.

In [ ]:
import subprocess
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'notebook', '--execute',
     'notebooks/05_Embedding_Clustering.ipynb', '--output', '05_Embedding_Clustering.ipynb'],
    capture_output=True, text=True, encoding='utf-8')
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## Tổng kết training

Sau khi 5 stages chạy xong:
- `clean_data/` chứa 2 file CSV đã clean (17 cột)
- `features/` chứa sparse X + dense SVD + SBERT embeddings + transformers
- `models/` chứa 2 model KMeans (TF-IDF + SBERT) + GMM + labels + profile + cluster names + demo

→ Sang `testing.ipynb` để demo nhanh trên test data (< 1 phút) — đây là notebook GV sẽ chạy lúc bảo vệ.